In [ ]:
import pandas as pd

In [ ]:
#IN_PATH = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6\1556_with_eligibility_title_abstract_495_86.xlsx'
#IN_PATH = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\tables\ris_with_decision.xlsx'
IN_PATH = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\organized\final_gpt_gemini_6_1.xlsx'


In [ ]:
df = pd.read_excel(IN_PATH)
df.head()

In [ ]:
lower_cols = [col.lower() for col in df.columns]
assert 'gpt' in lower_cols
assert 'golden' in lower_cols

In [ ]:
gpt_col = [col for col in df.columns if col.lower() == 'gpt'][0]
golden_col = [col for col in df.columns if col.lower() == 'golden'][0]
gpt_col, golden_col

In [ ]:
df = df[[gpt_col, golden_col]]
df.columns = ['GPT', 'golden']
df.head()

In [ ]:
counts = df.groupby(['GPT', 'golden']).golden.count().rename('count').reset_index()
better_names = {0: 'exclude', 1: 'include', False: 'exclude', True: 'include'}
counts['GPT'] = counts['GPT'].map(better_names.get)
counts['golden'] = counts['golden'].map(better_names.get)
counts

In [ ]:
table = pd.pivot(counts, columns='golden', index='GPT', values='count').fillna(0)
table = table.loc[['include', 'exclude'], ['include', 'exclude']]
table.loc['total', :] = table.loc['include', :] + table.loc['exclude', :]
table.loc[:, 'total'] = table.loc[:, 'include'] + table.loc[:, 'exclude']
table.style.format(precision=0)

In [ ]:
tp = table.loc['include', 'include']
fp = table.loc['include', 'exclude']
fn = table.loc['exclude', 'include']
stats = {
    'precision': tp / (tp + fp),
    'recall': tp / (tp + fn),
}
stats['f1'] = 2 / (1 / stats['precision'] + 1 / stats['recall'])
stats = pd.Series(stats, name='') * 100
stats.to_frame().style.format('{:.1f}%')

##### 